# Numerosity-PRF R² across scanners

Loads the per-(subject, session) R² maps produced by `balgrist/encoding_model/fit_mapper.py` and compares their distributions across the four scanner sessions.

Sessions: `ses-balgrist3t`, `ses-balgrist7t`, `ses-ibt7t`, `ses-sns3t`.

Voxels outside the fitting mask have R² = 0 exactly; we drop them with `r2 != 0`, which is equivalent to a brain-mask filter without needing the masks locally.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import nibabel as nib
import numpy as np
import pandas as pd
import seaborn as sns

BIDS_FOLDER = Path('~/data/ds-balgrist').expanduser()
# BIDS_FOLDER = Path('/shares/zne.uzh/gdehol/ds-balgrist')   # on cluster
DERIV = 'encoding_model'
SESSIONS = ('balgrist3t', 'balgrist7t', 'ibt7t', 'sns3t')

In [ ]:
def load_r2(subject, session, bids_folder=BIDS_FOLDER, deriv=DERIV):
    fn = (bids_folder / 'derivatives' / deriv
          / f'sub-{subject}' / f'ses-{session}' / 'func'
          / f'sub-{subject}_ses-{session}_task-mapper_desc-r2_space-T1w_pars.nii.gz')
    if not fn.exists():
        return None
    data = nib.load(str(fn)).get_fdata().ravel()
    return data[data != 0]


subjects = sorted({p.name.split('sub-')[1]
                   for p in (BIDS_FOLDER / 'derivatives' / DERIV).iterdir()
                   if p.is_dir() and p.name.startswith('sub-')})
print('subjects:', subjects)

rows = []
for sub in subjects:
    for ses in SESSIONS:
        r2 = load_r2(sub, ses)
        if r2 is None:
            print(f'  missing: sub-{sub} ses-{ses}')
            continue
        rows.append(pd.DataFrame({
            'subject': sub, 'session': ses,
            'field_strength': '7T' if '7t' in ses else '3T',
            'r2': r2,
        }))

df = pd.concat(rows, ignore_index=True)
df.groupby(['subject', 'session']).size()

## Whole-brain R² distribution per session

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
sns.violinplot(data=df[df['r2'] > 0], x='session', y='r2',
               hue='field_strength', ax=ax,
               order=['balgrist3t', 'sns3t', 'balgrist7t', 'ibt7t'],
               inner='quartile', cut=0)
ax.set_yscale('log')
ax.set_ylabel('R² (voxels with R² > 0)')
ax.set_title('Numerosity-PRF R² per scanner session')
plt.tight_layout()

## Tail mass: fraction of brain voxels above thresholds

In [ ]:
thresholds = [0.01, 0.02, 0.05, 0.10, 0.20]
tail_rows = []
for (sub, ses), g in df.groupby(['subject', 'session']):
    for t in thresholds:
        tail_rows.append({'subject': sub, 'session': ses, 'threshold': t,
                          'frac': float((g['r2'] > t).mean())})
tail = pd.DataFrame(tail_rows)

fig, axes = plt.subplots(1, len(thresholds), figsize=(4 * len(thresholds), 4),
                         sharey=True)
for ax, t in zip(axes, thresholds):
    sns.barplot(data=tail[tail['threshold'] == t], x='session', y='frac', ax=ax,
                order=['balgrist3t', 'sns3t', 'balgrist7t', 'ibt7t'],
                errorbar='sd')
    ax.set_title(f'R² > {t}')
    ax.set_xlabel('')
    ax.tick_params(axis='x', rotation=45)
axes[0].set_ylabel('fraction of voxels')
plt.tight_layout()

## Paired within-subject Balgrist 3T vs 7T

In [ ]:
summary = (df.groupby(['subject', 'session'])
             .agg(median_r2=('r2', 'median'),
                  mean_r2=('r2', 'mean'),
                  frac_above_02=('r2', lambda s: float((s > 0.02).mean())),
                  frac_above_10=('r2', lambda s: float((s > 0.10).mean())))
             .reset_index())
print(summary.pivot(index='subject', columns='session',
                    values='frac_above_02').round(3))

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
for ax, col, title in [(axes[0], 'median_r2', 'median R²'),
                        (axes[1], 'frac_above_02', 'fraction R² > 0.02')]:
    wide = summary.pivot(index='subject', columns='session', values=col)
    keep = ['balgrist3t', 'balgrist7t']
    wide = wide[keep].dropna()
    for sub_id, row in wide.iterrows():
        ax.plot(keep, row.values, marker='o', label=f'sub-{sub_id}')
    ax.set_title(title)
    ax.set_xlabel('')
axes[0].legend(fontsize=8, loc='best')
plt.tight_layout()

## Print summary table

In [ ]:
print('\n=== median R² per session ===')
print(summary.pivot(index='subject', columns='session',
                    values='median_r2').round(4))
print('\n=== fraction of voxels with R² > 0.02 ===')
print(summary.pivot(index='subject', columns='session',
                    values='frac_above_02').round(3))
print('\n=== fraction of voxels with R² > 0.10 ===')
print(summary.pivot(index='subject', columns='session',
                    values='frac_above_10').round(3))